In [1]:
import pandas as pd

In [ ]:
import pickle
from utils import take_all_bending_setups

with open('../../data/experiments_process_and_results.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)


In [ ]:
all_bending_setups = take_all_bending_setups(loaded_dict)
experiment_number = 60
experiment_as_dictinary = loaded_dict.get(f'Exp_{experiment_number}')
for idx, key in enumerate(list(experiment_as_dictinary.keys())):
    print(f"{idx+1}-{key}")
    

1-geometry_data_key_characteristics_arc
2-geometry_data_key_characteristics_linear_1
3-geometry_data_key_characteristics_linear_2
4-geometry_data_stl_suitable_arc
5-geometry_data_stl_suitable_linear_1
6-geometry_data_stl_suitable_linear_2
7-process_parameters_loads_machine
8-process_parameters_loads_sensor
9-process_parameters_movements
10-bending_setups


In [ ]:
df_machine = experiment_as_dictinary['process_parameters_movements']

In [ ]:
df_sensor = experiment_as_dictinary['process_parameters_loads_sensor']

In [ ]:
df_movement = experiment_as_dictinary['process_parameters_loads_machine']

In [ ]:
# Ensure 'Time_[s]' is the index for resampling
df_machine['Time_[s]'] = df_machine['Time_[s]'].astype(float)
df_machine.set_index('Time_[s]', inplace=True)
df_machine = df_machine.apply(pd.to_numeric, errors='coerce')
df_sensor['Time_[s]'] = df_sensor['Time_[s]'].astype(float)
df_sensor.set_index('Time_[s]', inplace=True)
df_sensor = df_sensor.apply(pd.to_numeric, errors='coerce')
df_movement['Time_[s]'] = df_movement['Time_[s]'].astype(float)
df_movement.set_index('Time_[s]', inplace=True)
df_movement = df_movement.apply(pd.to_numeric, errors='coerce')

In [ ]:
# Determine the common time range
start_time = max(df.index.min() for df in [df_machine, df_sensor, df_movement])
end_time = min(df.index.max() for df in [df_machine, df_sensor, df_movement])

In [ ]:
import numpy as np  # <--- import numpy

# Create a common time index with desired frequency (0.01s)
time_index = pd.Index(np.arange(start_time, end_time + 0.01, 0.01), name='Time_[s]')  # use np.arange

In [ ]:
# Reindex and interpolate missing values
df_machine_resampled = df_machine.reindex(time_index).interpolate(method='linear')
df_sensor_resampled = df_sensor.reindex(time_index).interpolate(method='linear')
df_movement_resampled = df_movement.reindex(time_index).interpolate(method='linear')

# Concatenate all datasets along columns
df_process_parameters = pd.concat([df_machine_resampled, df_sensor_resampled, df_movement_resampled], axis=1)

# Reset index if you want Time_[s] as a column again
df_process_parameters.reset_index(inplace=True)
df_process_parameters.index = df_process_parameters['Time_[s]']
df_process_parameters.drop(columns=['Time_[s]'], inplace = True)

In [ ]:
df_process_parameters.head()

,BEND-DIE_LATERAL_Movement_[mm],BEND-DIE_ROTATING_Angle_[°],BEND-DIE_VERTICAL_Movement_[mm],CLAMP-DIE_LATERAL_Movement_[mm],COLLET_AXIAL_Movement_[mm],COLLET_ROTATING_Movement_[mm],MANDREL_AXIAL_Movement_[mm],PRESSURE-DIE_AXIAL_Movement_[mm],PRESSURE-DIE_LATERAL_Movement_[mm],PRESSURE-DIE_LEFT_AXIAL_Movement_[mm],...,MACHINE_BEND-DIE_LATERAL_Max_Torque_[%],MACHINE_BEND-DIE_ROTATING_Max_Torque_[%],MACHINE_BEND-DIE_VERTICAL_Max_Torque_[%],MACHINE_CLAMP-DIE_LATERAL_Max_Torque_[%],MACHINE_COLLET_AXIAL_Max_Torque_[%],MACHINE_COLLET_ROTATING_Max_Torque_[%],MACHINE_MANDREL_AXIAL_Max_Torque_[%],MACHINE_PRESSURE-DIE_AXIAL_Max_Torque_[%],MACHINE_PRESSURE-DIE_LATERAL_Max_Torque_[%],MACHINE_PRESSURE-DIE_LEFT_AXIAL_Max_Torque_[%]
Time_[s],,,,,,,,,,,,,,,,,,,,,
0.00,-33.0,90.0,19.75,150.0,-289.999908,209.880402,-2907.600098,-4.8,-20.0,-7.5,...,0.474934,0.429286,-11.777778,1.375,-1.009333,2.5,0.526316,0.0,11.868002,0.3
0.01,-33.0,90.0,19.75,150.0,-289.999927,209.880402,-2907.600098,-4.8,-20.0,-7.5,...,0.474934,0.429572,-11.777778,1.375,-1.009200,2.5,0.526316,0.0,11.849802,0.3
0.02,-33.0,90.0,19.75,150.0,-289.999945,209.880402,-2907.600098,-4.8,-20.0,-7.5,...,0.474934,0.429857,-11.777778,1.375,-1.009066,2.5,0.526316,0.0,11.831602,0.3
0.03,-33.0,90.0,19.75,150.0,-289.999963,209.880402,-2907.600098,-4.8,-20.0,-7.5,...,0.474934,0.430143,-11.777778,1.375,-1.008933,2.5,0.526316,0.0,11.813402,0.3
0.04,-33.0,90.0,19.75,150.0,-289.999982,209.880402,-2907.600098,-4.8,-20.0,-7.5,...,0.474934,0.430429,-11.777778,1.375,-1.008800,2.5,0.526316,0.0,11.795202,0.3


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_numeric_df(df, title="DataFrame Plot", figsize=(10, 6)):
    """
    Plots all numeric columns of a DataFrame using the index as x-axis.
    
    Parameters:
    - df: pd.DataFrame, the DataFrame to plot
    - title: str, the title of the plot
    - figsize: tuple, size of the figure
    """
    # Convert all columns to numeric, non-convertible values become NaN
    numeric_df = df.apply(pd.to_numeric, errors='coerce')
    
    # Plot all numeric columns
    ax = numeric_df.plot(figsize=figsize, marker='o')
    ax.set_title(title)
    ax.set_xlabel("Index")
    ax.set_ylabel("Values")
    ax.grid(True)
    plt.legend(title="Columns")
    plt.show()


In [ ]:
df_geometry_data = experiment_as_dictinary['geometry_data_key_characteristics_arc'].drop(columns= ['Tube_Section'])
df_geometry_data.index = df_geometry_data['Angle[degree]ORDistance[mm]']
df_geometry_data.drop(columns=['Angle[degree]ORDistance[mm]'], inplace=True)
df_geometry_data = df_geometry_data.apply(pd.to_numeric, errors='coerce')
df_geometry_data.tail()

,Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
Angle[degree]ORDistance[mm],,,,
41.00,22.150528,21.334929,0.037020,-0.696629
42.00,22.144354,21.339386,0.036537,-0.692172
43.00,22.135644,21.339377,0.036142,-0.692181
44.00,22.128793,21.341037,0.035756,-0.690521
44.71,22.127360,21.345171,0.035503,-0.686387


In [ ]:
# Suppose you want angles between 10 and 30 degrees
start_angle = 30
end_angle = 40

# Select rows using index (which is Angle[degree])
window_df = df_geometry_data.loc[start_angle:end_angle]
window_df


,Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
Angle[degree]ORDistance[mm],,,,
30.0,22.184541,21.227649,0.043433,-0.803909
31.0,22.188014,21.235055,0.043254,-0.796503
32.0,22.188424,21.242944,0.042915,-0.788614
33.0,22.188525,21.251329,0.042539,-0.780229
34.0,22.187959,21.260621,0.042091,-0.770938
35.0,22.184515,21.271308,0.041450,-0.760250
36.0,22.179543,21.281391,0.040767,-0.750167
37.0,22.173055,21.294676,0.039869,-0.736882
38.0,22.166394,21.302575,0.039208,-0.728984
